# UAS Praktikum Machine Learning 2026
## Notebook 1 - EDA & Preprocessing
### Studi Kasus: Klasifikasi Risiko Klaim Pengemudi (Telematics Asuransi)

**Kelompok:** XXX | **Jalur:** Klasifikasi + Mobile (Flutter)

Dataset telematics sintetis (So et al., 2021) berisi 100.000 pengemudi dengan 50 fitur
(data tradisional + telematics). Tujuan: memprediksi apakah seorang pengemudi **berisiko**
(pernah mengajukan klaim) atau tidak.

Referensi metodologi: McDonnell et al. (2023), *Deep learning in insurance: Accuracy and
model interpretability using TabNet*, Expert Systems With Applications 217.

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
sns.set_style("whitegrid")

df = pd.read_csv("../data/telematics.csv")
print("Shape:", df.shape)
df.head()

### 1. Definisi Target
Paper mendefinisikan `ClaimYN = 1` jika `NB_Claim > 1 & AMT_Claim > 1000`. Namun pada versi
dataset ini aturan tersebut hanya menghasilkan 178 sampel positif (0.18%) yang mustahil
dipelajari model. Maka kami memakai definisi yang lebih standar dan dapat dipelajari:

> **ClaimYN = 1 jika NB_Claim >= 1** (pengemudi pernah mengajukan klaim = berisiko).

In [ ]:
df["ClaimYN"] = (df["NB_Claim"] >= 1).astype(int)
print(df["ClaimYN"].value_counts())
print(f"Proporsi positif: {100*df['ClaimYN'].mean():.2f}%  (DATA SANGAT IMBALANCED)")

fig, ax = plt.subplots(figsize=(5,4))
vc = df["ClaimYN"].value_counts().sort_index()
b = ax.bar(["Tidak Berisiko (0)","Berisiko (1)"], vc.values, color=["#94a3b8","#dc2626"])
ax.bar_label(b, fmt="%d"); ax.set_title("Distribusi Kelas Target (Imbalanced)")
plt.tight_layout(); plt.show()

**Insight kritis:** Karena hanya ~4% data positif, metrik **akurasi menyesatkan** -
model yang selalu menebak "tidak berisiko" sudah mendapat ~96% akurasi. Maka evaluasi
wajib memakai F1, Recall, AUC, dan MCC, bukan akurasi.

In [ ]:
# Pengecekan kualitas data
print("Missing values:", df.isnull().sum().sum())
print("Kolom kategorikal:", [c for c in df.columns if df[c].dtype=='object'])
# Distribusi beberapa fitur telematics kunci
fig, ax = plt.subplots(1,3, figsize=(13,3.5))
for a, col in zip(ax, ["Total.miles.driven","Annual.pct.driven","Brake.14miles"]):
    sns.histplot(df[col], bins=40, ax=a, color="#2563eb"); a.set_title(col)
plt.tight_layout(); plt.show()

### 2. Preprocessing & Data Splitting
- One-hot encoding untuk 4 kolom kategorikal (Insured.sex, Marital, Car.use, Region)
- **Anti-leakage:** kolom `NB_Claim` & `AMT_Claim` DIBUANG dari fitur (mendefinisikan target)
- Standardisasi (StandardScaler) di-*fit* hanya pada data train
- Split stratified 70% train / 15% validation / 15% test

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import json, os

SEED = 42
LEAK = ["NB_Claim","AMT_Claim"]; TARGET = "ClaimYN"
cat_cols = ["Insured.sex","Marital","Car.use","Region"]
num_cols = [c for c in df.columns if c not in cat_cols + LEAK + [TARGET]]

df_enc = pd.get_dummies(df[cat_cols], prefix=cat_cols)
X_all = pd.concat([df[num_cols], df_enc], axis=1).astype("float32")
y_all = df[TARGET].values
feat_names = X_all.columns.tolist()

X_tr, X_tmp, y_tr, y_tmp = train_test_split(X_all, y_all, test_size=.30, stratify=y_all, random_state=SEED)
X_va, X_te, y_va, y_te = train_test_split(X_tmp, y_tmp, test_size=.50, stratify=y_tmp, random_state=SEED)

scaler = StandardScaler().fit(X_tr[num_cols])
def scale(X):
    X = X.copy(); X[num_cols] = scaler.transform(X[num_cols]); return X.values.astype("float32")
Xtr, Xva, Xte = scale(X_tr), scale(X_va), scale(X_te)
print("train/val/test:", Xtr.shape, Xva.shape, Xte.shape)

os.makedirs("../outputs/models", exist_ok=True)
np.savez("../outputs/processed.npz", Xtr=Xtr, Xva=Xva, Xte=Xte, ytr=y_tr, yva=y_va, yte=y_te)
# simpan spec preprocessing untuk aplikasi Flutter
prep = {"feature_order": feat_names, "numeric_features": num_cols,
        "scaler_mean": dict(zip(num_cols, scaler.mean_.tolist())),
        "scaler_scale": dict(zip(num_cols, scaler.scale_.tolist())),
        "categorical_values": {c: sorted(df[c].unique().tolist()) for c in cat_cols}}
json.dump(prep, open("../outputs/models/preprocessing.json","w"), indent=2)
print("Data terproses & spec preprocessing tersimpan.")